# NeuraSight — Chest X-Ray Model Training V2

**Improved pipeline** fixing accuracy issues from V1 (was 75%, targeting 90%+).

Key improvements:
- Class weighting for imbalanced dataset
- 30 epochs with early stopping (patience=7)
- Lower LR (5e-5) with warmup + cosine annealing
- No ColorJitter (medical images)
- Label smoothing (0.1)
- WeightedRandomSampler for balanced batches
- 3 models: EfficientNet-B0, ResNet-50, DenseNet-121

In [ ]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

# Set Kaggle token
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0ba6655de23e518c18090f7b10ff882e'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('KGAT_0ba6655de23e518c18090f7b10ff882e')
os.chmod('/root/.kaggle/access_token', 0o600)

# Download dataset to local SSD
LOCAL_DATA = '/content/chest_xray'
if not os.path.isdir(os.path.join(LOCAL_DATA, 'train')):
    print('Downloading dataset...')
    !kaggle datasets download -d muhammadrehan00/chest-xray-dataset -p /content/ --force
    print('Extracting...')
    with zipfile.ZipFile('/content/chest-xray-dataset.zip', 'r') as zf:
        zf.extractall(LOCAL_DATA)
    os.remove('/content/chest-xray-dataset.zip')
    print('\u2713 Done')
else:
    print('\u2713 Dataset already on local disk')

!pip install timm seaborn -q

In [ ]:
import os, json, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, accuracy_score
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
TRAIN_DIR = '/content/chest_xray/train'
VAL_DIR = '/content/chest_xray/val'
TEST_DIR = '/content/chest_xray/test'
SAVE_DIR = '/content/drive/MyDrive/NeuraSight/models'
REPORTS_DIR = '/content/drive/MyDrive/NeuraSight/reports/chest_xray'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MODEL_CONFIG = {
    "efficientnet": {"timm_name": "efficientnet_b0", "save_name": "CHEST_XRAY_EFFICIENTNET"},
    "resnet": {"timm_name": "resnet50", "save_name": "CHEST_XRAY_RESNET"},
    "densenet": {"timm_name": "densenet121", "save_name": "CHEST_XRAY_DENSENET"},
}
MODELS_TO_TRAIN = ["efficientnet", "resnet", "densenet"]
NUM_CLASSES = 3
CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]

# Training hyperparameters (FIXED from V1)
EPOCHS = 30
PATIENCE = 7  # early stopping
LR = 5e-5  # lower than V1's 1e-4
WEIGHT_DECAY = 1e-3  # lower than V1's 0.01
BATCH_SIZE = 32
LABEL_SMOOTHING = 0.1

print(f'Epochs: {EPOCHS}, Patience: {PATIENCE}')
print(f'LR: {LR}, WD: {WEIGHT_DECAY}, BS: {BATCH_SIZE}')
print(f'Label smoothing: {LABEL_SMOOTHING}')

In [ ]:
# Transforms — NO ColorJitter (medical images!)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = ImageFolder(TEST_DIR, transform=val_transform)

print(f'Classes: {train_dataset.classes}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# === FIX #1: Compute class weights for imbalanced dataset ===
class_counts = np.bincount([label for _, label in train_dataset.samples])
print(f'Class counts: {dict(zip(train_dataset.classes, class_counts))}')

# Inverse frequency weighting
total = sum(class_counts)
class_weights = torch.tensor([total / (NUM_CLASSES * c) for c in class_counts], dtype=torch.float32).to(device)
print(f'Class weights: {class_weights.cpu().numpy()}')

# === FIX #2: WeightedRandomSampler for balanced batches ===
sample_weights = [1.0 / class_counts[label] for _, label in train_dataset.samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders (sampler replaces shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100.0 * correct / total

In [ ]:
def train_full(model_key):
    """Train one model with all V2 fixes applied."""
    cfg = MODEL_CONFIG[model_key]
    save_name = cfg['save_name']
    best_path = os.path.join(SAVE_DIR, save_name + '.pth')

    print(f'\n{"="*60}')
    print(f'TRAINING: {model_key} ({cfg["timm_name"]})')
    print(f'{"="*60}')

    # Create model
    model = timm.create_model(cfg['timm_name'], pretrained=True, num_classes=NUM_CLASSES).to(device)

    # === FIX #3: Class-weighted loss with label smoothing ===
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

    # === FIX #4: Lower LR ===
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # === FIX #5: Warmup + Cosine scheduler ===
    # 3 epochs warmup then cosine decay
    warmup_epochs = 3
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Training loop with early stopping
    best_val_acc = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    start = time.time()
    for epoch in range(EPOCHS):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        marker = ''
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
            marker = f' \u2190 saved ({best_val_acc:.2f}%)'
        else:
            patience_counter += 1

        current_lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch+1:2d}/{EPOCHS} | Train {tr_acc:.2f}% | Val {vl_acc:.2f}% | LR {current_lr:.2e}{marker}')

        # Early stopping
        if patience_counter >= PATIENCE:
            print(f'  \u26a0 Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
            break

    elapsed = time.time() - start
    print(f'  Training complete in {elapsed/60:.1f} min')
    print(f'  Best Val Acc: {best_val_acc:.2f}%')

    # Plot curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
    ax1.set_title(f'{model_key} Loss'); ax1.legend(); ax1.set_xlabel('Epoch')
    ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
    ax2.set_title(f'{model_key} Accuracy'); ax2.legend(); ax2.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, f'{save_name}_v2_curves.png'), dpi=150)
    plt.show()

    # Generate test probabilities
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    probs_list, y_true, y_pred = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            probs_list.append(probs.cpu().numpy())
            y_pred.extend(probs.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())

    probs_arr = np.concatenate(probs_list, axis=0)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Save
    np.save(os.path.join(SAVE_DIR, save_name + '_test_probs.npy'), probs_arr)
    np.save(os.path.join(SAVE_DIR, 'test_labels.npy'), y_true)

    # Metrics
    acc = accuracy_score(y_true, y_pred) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    metrics = {
        'model': model_key, 'accuracy': round(acc, 2),
        'precision': round(prec*100, 2), 'recall': round(rec*100, 2),
        'f1': round(f1*100, 2), 'best_val_acc': round(best_val_acc, 2),
    }
    with open(os.path.join(SAVE_DIR, save_name + '_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f'\n  TEST Results: Acc={acc:.2f}% P={prec*100:.1f}% R={rec*100:.1f}% F1={f1*100:.1f}%')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_key} Confusion Matrix (V2)'); plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, f'{save_name}_v2_cm.png'), dpi=150)
    plt.show()

    del model; gc.collect(); torch.cuda.empty_cache()
    return metrics

In [ ]:
all_metrics = []
for model_key in MODELS_TO_TRAIN:
    metrics = train_full(model_key)
    all_metrics.append(metrics)

print('\n' + '='*60)
print('ALL MODELS COMPLETE')
print('='*60)
df = pd.DataFrame(all_metrics)
print(df[['model', 'accuracy', 'precision', 'recall', 'f1', 'best_val_acc']].to_string(index=False))

In [ ]:
from google.colab import files
for key in MODELS_TO_TRAIN:
    sn = MODEL_CONFIG[key]['save_name']
    p = os.path.join(SAVE_DIR, sn + '.pth')
    if os.path.exists(p):
        files.download(p)
        print(f'\u2713 {sn}.pth')
print('\nDone! Run Chest_Xray_Stacking_Ensemble.ipynb next.')